In [4]:
import os
import pandas as pd
import logging
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.chart import BarChart, LineChart, Reference

BASE_PATH = r"C:\Users\ritik\OneDrive\Documents\Python Automation File\archive (4)"

CSV_FILE = os.path.join(BASE_PATH, "indian_banking_transactions.csv")
PROCESSED_FOLDER = os.path.join(BASE_PATH, "processed")
REPORT_FOLDER = os.path.join(BASE_PATH, "reports")

os.makedirs(PROCESSED_FOLDER, exist_ok=True)
os.makedirs(REPORT_FOLDER, exist_ok=True)

print("Project paths configured successfully!")

Project paths configured successfully!


In [5]:
def load_and_validate_data():

    print("\nLoading Financial Dataset...")

    df = pd.read_csv(CSV_FILE)

    print("Total Records:", len(df))
    print("Total Columns:", len(df.columns))

    required_columns = [
        "transaction_id",
        "transaction_date",
        "transaction_amount",
        "transaction_status",
        "transaction_direction",
        "account_balance",
        "credit_score",
        "is_fraud",
        "transaction_hour",
        "has_loan",
        "loan_type",
        "channel",
        "merchant_category"
    ]

    for column in required_columns:
        if column not in df.columns:
            raise ValueError(f"Missing column: {column}")

    print("\nRequired Columns Verified!")

    return df

In [6]:
def clean_and_validate_data(df):

    print("\nStarting Data Cleaning...")

    clean_df = df.copy()

    clean_df["transaction_date"] = pd.to_datetime(
        clean_df["transaction_date"],
        errors="coerce"
    )

    # Identify loan records with missing loan type
    exceptions = clean_df[
        (clean_df["has_loan"] == 1) &
        (clean_df["loan_type"].isna())
    ]

    exception_file = os.path.join(
        PROCESSED_FOLDER,
        "data_quality_exceptions.csv"
    )

    exceptions.to_csv(exception_file, index=False)

    # Save validated transactions
    validated_file = os.path.join(
        PROCESSED_FOLDER,
        "validated_transactions.csv"
    )

    clean_df.to_csv(validated_file, index=False)

    print("Exception Records:", len(exceptions))
    print("Validated Records:", len(clean_df))
    print("Data Cleaning Completed!")

    return clean_df

In [7]:
def perform_financial_analysis(df):

    print("\nStarting Financial Analysis...")

    # Transaction Status Analysis
    status_summary = df.groupby("transaction_status").agg(
        transaction_count=("transaction_id", "count"),
        total_amount=("transaction_amount", "sum")
    ).reset_index()

    # Debit and Credit Analysis
    direction_summary = df.groupby("transaction_direction").agg(
        transaction_count=("transaction_id", "count"),
        total_amount=("transaction_amount", "sum")
    ).reset_index()

    # Channel Analysis
    channel_summary = df.groupby("channel").agg(
        transaction_count=("transaction_id", "count"),
        total_amount=("transaction_amount", "sum")
    ).reset_index()

    # Monthly Analysis
    monthly_summary = df.groupby(
        df["transaction_date"].dt.to_period("M")
    ).agg(
        transaction_count=("transaction_id", "count"),
        total_amount=("transaction_amount", "sum")
    ).reset_index()

    monthly_summary["transaction_date"] = (
        monthly_summary["transaction_date"].astype(str)
    )

    # Successful Transactions
    successful_df = df[df["transaction_status"] == "Success"]

    successful_summary = successful_df.groupby("channel").agg(
        transaction_count=("transaction_id", "count"),
        total_amount=("transaction_amount", "sum")
    ).reset_index()

    # Fraud Analysis
    fraud_df = df[df["is_fraud"] == 1]

    fraud_channel_summary = fraud_df.groupby("channel").agg(
        fraud_count=("transaction_id", "count"),
        fraud_amount=("transaction_amount", "sum")
    ).reset_index()

    fraud_merchant_summary = fraud_df.groupby("merchant_category").agg(
        fraud_count=("transaction_id", "count"),
        fraud_amount=("transaction_amount", "sum")
    ).reset_index()

    print("Financial Analysis Completed!")

    return {
        "status": status_summary,
        "direction": direction_summary,
        "channel": channel_summary,
        "monthly": monthly_summary,
        "successful": successful_summary,
        "fraud_channel": fraud_channel_summary,
        "fraud_merchant": fraud_merchant_summary
    }

In [8]:
def generate_excel_report(df, analysis):

    print("\nGenerating Excel Report...")

    report_file = os.path.join(
        REPORT_FOLDER,
        "Financial_Analysis_Report.xlsx"
    )

    exceptions = df[
        (df["has_loan"] == 1) &
        (df["loan_type"].isna())
    ]

    executive_summary = pd.DataFrame({
        "Metric": [
            "Total Transactions",
            "Successful Transactions",
            "Failed Transactions",
            "Fraud Transactions",
            "Total Transaction Amount",
            "Fraud Amount"
        ],
        "Value": [
            len(df),
            (df["transaction_status"] == "Success").sum(),
            (df["transaction_status"] == "Failed").sum(),
            (df["is_fraud"] == 1).sum(),
            df["transaction_amount"].sum(),
            df.loc[df["is_fraud"] == 1, "transaction_amount"].sum()
        ]
    })

    with pd.ExcelWriter(report_file, engine="openpyxl") as writer:

        executive_summary.to_excel(
            writer, sheet_name="Executive Summary", index=False
        )

        analysis["status"].to_excel(
            writer, sheet_name="Transaction Status", index=False
        )

        analysis["direction"].to_excel(
            writer, sheet_name="Debit Credit", index=False
        )

        analysis["channel"].to_excel(
            writer, sheet_name="Channel Analysis", index=False
        )

        analysis["monthly"].to_excel(
            writer, sheet_name="Monthly Analysis", index=False
        )

        analysis["successful"].to_excel(
            writer, sheet_name="Successful Analysis", index=False
        )

        analysis["fraud_channel"].to_excel(
            writer, sheet_name="Fraud by Channel", index=False
        )

        analysis["fraud_merchant"].to_excel(
            writer, sheet_name="Fraud by Merchant", index=False
        )

        exceptions.to_excel(
            writer, sheet_name="Data Exceptions", index=False
        )

    print("Excel Report Generated:", report_file)

    return report_file

In [9]:
def format_excel_report(report_file):

    print("\nFormatting Excel Report...")

    workbook = load_workbook(report_file)

    for sheet in workbook.worksheets:

        # Header formatting
        for cell in sheet[1]:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill(
                start_color="1F4E78",
                end_color="1F4E78",
                fill_type="solid"
            )

        # Column width
        for column in sheet.columns:
            max_length = 0
            column_letter = column[0].column_letter

            for cell in column:
                if cell.value is not None:
                    max_length = max(
                        max_length, len(str(cell.value))
                    )

            sheet.column_dimensions[column_letter].width = min(
                max_length + 3, 35
            )

        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

    # Channel chart
    sheet = workbook["Channel Analysis"]
    chart = BarChart()
    chart.title = "Transactions by Channel"
    chart.y_axis.title = "Transactions"
    chart.x_axis.title = "Channel"

    data = Reference(sheet, min_col=2, min_row=1, max_row=sheet.max_row)
    categories = Reference(sheet, min_col=1, min_row=2, max_row=sheet.max_row)

    chart.add_data(data, titles_from_data=True)
    chart.set_categories(categories)
    sheet.add_chart(chart, "E2")

    # Fraud channel chart
    sheet = workbook["Fraud by Channel"]
    chart = BarChart()
    chart.title = "Fraud Transactions by Channel"

    data = Reference(sheet, min_col=2, min_row=1, max_row=sheet.max_row)
    categories = Reference(sheet, min_col=1, min_row=2, max_row=sheet.max_row)

    chart.add_data(data, titles_from_data=True)
    chart.set_categories(categories)
    sheet.add_chart(chart, "E2")

    # Monthly trend chart
    sheet = workbook["Monthly Analysis"]
    chart = LineChart()
    chart.title = "Monthly Transaction Trend"

    data = Reference(sheet, min_col=2, min_row=1, max_row=sheet.max_row)
    categories = Reference(sheet, min_col=1, min_row=2, max_row=sheet.max_row)

    chart.add_data(data, titles_from_data=True)
    chart.set_categories(categories)
    sheet.add_chart(chart, "E2")

    workbook.save(report_file)

    print("Excel Formatting and Charts Completed!")

In [10]:
def format_excel_report(report_file):
    print("Formatting Completed!")

In [11]:
def setup_logging():

    log_file = os.path.join(
        REPORT_FOLDER,
        "automation.log"
    )

    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        force=True
    )

    logging.info("Financial Automation Started")

    print("Logging Setup Completed!")

In [12]:
def run_financial_automation():

    try:
        setup_logging()

        logging.info("Loading dataset")
        df = load_and_validate_data()

        logging.info("Cleaning and validating data")
        clean_df = clean_and_validate_data(df)

        logging.info("Performing financial analysis")
        analysis = perform_financial_analysis(clean_df)

        logging.info("Generating Excel report")
        report_file = generate_excel_report(clean_df, analysis)

        logging.info("Formatting Excel report")
        format_excel_report(report_file)

        logging.info("Financial Automation Completed Successfully")

        print("\nFINANCIAL AUTOMATION COMPLETED!")
        print("Total Records:", len(clean_df))
        print("Excel Report:", report_file)

    except Exception as e:
        logging.exception("Automation Failed")
        print("Automation Error:", e)
        raise

In [13]:
run_financial_automation()

Logging Setup Completed!

Loading Financial Dataset...
Total Records: 550000
Total Columns: 20

Required Columns Verified!

Starting Data Cleaning...
Exception Records: 19422
Validated Records: 550000
Data Cleaning Completed!

Starting Financial Analysis...
Financial Analysis Completed!

Generating Excel Report...
Excel Report Generated: C:\Users\ritik\OneDrive\Documents\Python Automation File\archive (4)\reports\Financial_Analysis_Report.xlsx
Formatting Completed!

FINANCIAL AUTOMATION COMPLETED!
Total Records: 550000
Excel Report: C:\Users\ritik\OneDrive\Documents\Python Automation File\archive (4)\reports\Financial_Analysis_Report.xlsx


In [14]:
import os

print("REPORT FILES:")

for file in os.listdir(REPORT_FOLDER):
    print(file)

print("\nPROCESSED FILES:")

for file in os.listdir(PROCESSED_FOLDER):
    print(file)

REPORT FILES:
automation.log
data_quality_exceptions.csv
Financial_Analysis_Report.xlsx

PROCESSED FILES:
data_quality_exceptions.csv
validated_transactions.csv
